In [5]:
%pip install pandas numpy scikit-learn xgboost lightgbm joblib catboost pytorch-tabnet tensorflow --quiet

import pandas as pd
import numpy as np
import joblib
import os

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score

RANDOM_STATE = 42
TEST_SIZE    = 0.20

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("All imports and directory setup successful.")

All imports and directory setup successful.


In [6]:
df = pd.read_csv("/content/diabetes_model_base.csv")

# ── Fix: pandas 2.0+ reads string columns as StringDtype by default.
# sklearn's ColumnTransformer requires object dtype for column name lookup.
CATEGORICAL_COLS_RAW = ["primary_diagnosis_group_reduced", "age_risk_group"]
for col in CATEGORICAL_COLS_RAW:
    df[col] = df[col].astype(object)

print("Shape         :", df.shape)
print("Null values   :", df.isna().sum().sum())
print("\nDtypes of categorical columns after fix:")
print(df[CATEGORICAL_COLS_RAW].dtypes)
print("\nTarget distribution:")
print(df["readmit_30"].value_counts(normalize=True).round(4))


Shape         : (101766, 40)
Null values   : 0

Dtypes of categorical columns after fix:
primary_diagnosis_group_reduced    object
age_risk_group                     object
dtype: object

Target distribution:
readmit_30
0    0.8884
1    0.1116
Name: proportion, dtype: float64


In [7]:
base_table = pd.read_csv(
    "/content/diabetic_data_base_table.csv",
    usecols=["patient_nbr"],
    low_memory=False
)

df["patient_nbr"] = base_table["patient_nbr"].values

patient_labels = (
    df.groupby("patient_nbr")["readmit_30"]
    .max()
    .reset_index()
)

train_patients, test_patients = train_test_split(
    patient_labels["patient_nbr"],
    test_size=TEST_SIZE,
    stratify=patient_labels["readmit_30"],
    random_state=RANDOM_STATE
)

train_df = df[df["patient_nbr"].isin(train_patients)].drop(columns=["patient_nbr"])
test_df  = df[df["patient_nbr"].isin(test_patients)].drop(columns=["patient_nbr"])
df.drop(columns=["patient_nbr"], inplace=True)

print(f"Train encounters : {len(train_df):,}  |  Positive rate: {train_df['readmit_30'].mean():.4f}")
print(f"Test  encounters : {len(test_df):,}   |  Positive rate: {test_df['readmit_30'].mean():.4f}")

Train encounters : 81,347  |  Positive rate: 0.1115
Test  encounters : 20,419   |  Positive rate: 0.1119


In [8]:
TARGET = "readmit_30"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("y_train positives:", y_train.sum(), "  y_test positives:", y_test.sum())

X_train: (81347, 39)   X_test: (20419, 39)
y_train positives: 9072   y_test positives: 2285


In [9]:
CATEGORICAL_COLS = [
    "primary_diagnosis_group_reduced",
    "age_risk_group"
]

NUMERIC_COLS = [c for c in X_train.columns if c not in CATEGORICAL_COLS]

print("Categorical features:", CATEGORICAL_COLS)
print("Numeric features    :", NUMERIC_COLS)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ("num", StandardScaler(), NUMERIC_COLS)
    ],
    remainder="drop"
)

print("\nPreprocessor template configured.")

Categorical features: ['primary_diagnosis_group_reduced', 'age_risk_group']
Numeric features    : ['age_ordinal', 'time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_emergency', 'number_outpatient', 'number_inpatient', 'prior_inpatient_flag', 'on_insulin', 'med_change_flag', 'diabetes_med_flag', 'medication_burden_bucket', 'total_visits', 'emergency_ratio', 'inpatient_ratio', 'visit_intensity', 'high_utilization', 'procedure_density', 'diagnosis_complexity', 'high_glucose_flag', 'high_A1C_flag', 'glu_level', 'a1c_level', 'has_circulatory', 'has_respiratory', 'has_diabetes_diag', 'num_unique_diag_groups', 'num_non_other_diag', 'num_active_medications', 'insulin_active', 'med_change_intensity', 'med_stable', 'meds_x_time', 'inpatient_x_meds', 'labs_x_time', 'age_x_meds', 'inpatient_x_time']

Preprocessor template configured.


In [10]:
neg_count        = (y_train == 0).sum()
pos_count        = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"Negative train samples : {neg_count:,}")
print(f"Positive train samples : {pos_count:,}")
print(f"scale_pos_weight       : {scale_pos_weight:.2f}")

Negative train samples : 72,275
Positive train samples : 9,072
scale_pos_weight       : 7.97


In [11]:
# -- 1. Logistic Regression --
pipeline_lr = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(class_weight="balanced", C=0.1, max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE))
])

# -- 2. Decision Tree --
pipeline_dt = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(max_depth=6, min_samples_leaf=100, class_weight="balanced", random_state=RANDOM_STATE))
])

# -- 3. Random Forest --
pipeline_rf = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=10, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))
])

# -- 4. HistGradientBoosting --
pipeline_hgb = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", HistGradientBoostingClassifier(max_iter=300, max_depth=6, learning_rate=0.05, class_weight="balanced", random_state=RANDOM_STATE))
])

# -- 5. XGBoost --
pipeline_xgb = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.03, scale_pos_weight=scale_pos_weight, eval_metric="aucpr", n_jobs=-1, random_state=RANDOM_STATE))
])

# -- 6. LightGBM --
pipeline_lgbm = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LGBMClassifier(n_estimators=400, max_depth=7, learning_rate=0.03, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE, verbose=-1))
])

# -- 7. ExtraTrees --
pipeline_et = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
])

# -- 8. GradientBoosting --
pipeline_gb = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))
])

# -- 9. SVM (Optimized with Subsampling for speed) --
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier

# Standard SVC is too slow for 80k rows. Using LinearSVC or subsampling is recommended.
# Here we use a 10k sample subset for the standard SVC to keep the non-linear capability while ensuring it finishes.
pipeline_svm = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", SVC(probability=True, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
])

# -- 10. CatBoost --
pipeline_cat = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, loss_function="Logloss", verbose=0, random_state=RANDOM_STATE))
])

print("All pipelines defined. SVM optimized with iteration limits.")

All pipelines defined. SVM optimized with iteration limits.


In [12]:
models = {
    "Logistic Regression"  : pipeline_lr,
    "Decision Tree"        : pipeline_dt,
    "Random Forest"        : pipeline_rf,
    "HistGradientBoosting" : pipeline_hgb,
    "XGBoost"              : pipeline_xgb,
    "LightGBM"             : pipeline_lgbm,
    "extra_trees"          : pipeline_et,
    "gradient_boosting"    : pipeline_gb,
    "svm"                  : pipeline_svm,
    "catboost"             : pipeline_cat
}

for name, pipeline in models.items():
    print(f"Training {name}...", end=" ", flush=True)
    if name == "svm":
        # Subsample specifically for SVM to prevent 1hr+ hang
        X_svm, _, y_svm, _ = train_test_split(X_train, y_train, train_size=10000, stratify=y_train, random_state=RANDOM_STATE)
        pipeline.fit(X_svm, y_svm)
    else:
        pipeline.fit(X_train, y_train)
    print("Done.")

print("\nAll registered models trained.")

Training Logistic Regression... Done.
Training Decision Tree... Done.
Training Random Forest... Done.
Training HistGradientBoosting... Done.
Training XGBoost... Done.
Training LightGBM... Done.
Training extra_trees... Done.
Training gradient_boosting... Done.
Training svm... Done.
Training catboost... 

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Done.

All registered models trained.


In [13]:
header = f"{'Model':<25} {'Train ROC-AUC':>15} {'Train F1 (pos)':>15}"
print(header)
print("-" * len(header))

for name, pipeline in models.items():
    y_prob = pipeline.predict_proba(X_train)[:, 1]
    y_pred = pipeline.predict(X_train)
    auc    = roc_auc_score(y_train, y_prob)
    report = classification_report(y_train, y_pred, output_dict=True)
    f1_pos = report["1"]["f1-score"]
    print(f"{name:<25} {auc:>15.4f} {f1_pos:>15.4f}")

print("\nNote: in-sample scores only. Held-out evaluation is in Notebook 06.")

Model                       Train ROC-AUC  Train F1 (pos)
---------------------------------------------------------
Logistic Regression                0.6425          0.2535
Decision Tree                      0.6440          0.2598
Random Forest                      0.7935          0.3799
HistGradientBoosting               0.6856          0.2782
XGBoost                            0.7108          0.2943


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM                           0.7410          0.3141
extra_trees                        1.0000          0.9998
gradient_boosting                  0.6568          0.0263
svm                                0.5696          0.2035
catboost                           0.6772          0.0427

Note: in-sample scores only. Held-out evaluation is in Notebook 06.


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# Transform raw X_train into numeric features using the preprocessor
X_train_transformed = preprocessor.transform(X_train)

dnn_model = Sequential([
    Input(shape=(X_train_transformed.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

dnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

dnn_model.fit(
    X_train_transformed,
    y_train,
    epochs=20,
    batch_size=256,
    verbose=1
)

dnn_probs = dnn_model.predict(X_train_transformed).ravel()

Epoch 1/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.5960 - loss: 0.3568
Epoch 2/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6201 - loss: 0.3424
Epoch 3/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - AUC: 0.6266 - loss: 0.3401
Epoch 4/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.6299 - loss: 0.3393
Epoch 5/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.6316 - loss: 0.3386
Epoch 6/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.6366 - loss: 0.3375
Epoch 7/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6376 - loss: 0.3372
Epoch 8/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6397 - loss: 0.3365
Epoch 9/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6419 - loss: 0.3360
Epoch 10/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6421 - loss: 0.3360
Epoch 11/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - AUC: 0.6427 - loss: 0.3357
Epoch 12/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6457 - loss: 0.3353
Epoch 13/20
318/318 ━━━━━

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# Fix: Transform raw X_train into numeric features using the preprocessor
X_train_transformed = preprocessor.transform(X_train)

dnn_model = Sequential([
    Input(shape=(X_train_transformed.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

dnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

dnn_model.fit(
    X_train_transformed,
    y_train,
    epochs=20,
    batch_size=256,
    verbose=1
)

dnn_probs = dnn_model.predict(X_train_transformed).ravel()

Epoch 1/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - AUC: 0.5972 - loss: 0.3525
Epoch 2/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.6196 - loss: 0.3425
Epoch 3/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - AUC: 0.6283 - loss: 0.3401
Epoch 4/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - AUC: 0.6330 - loss: 0.3387
Epoch 5/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6324 - loss: 0.3384
Epoch 6/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6373 - loss: 0.3372
Epoch 7/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6362 - loss: 0.3374
Epoch 8/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.6435 - loss: 0.3359
Epoch 9/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6446 - loss: 0.3355
Epoch 10/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6451 - loss: 0.3356
Epoch 11/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - AUC: 0.6445 - loss: 0.3356
Epoch 12/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6463 - loss: 0.3352
Epoch 13/20
318/318 ━━━━

In [19]:
from pytorch_tabnet.tab_model import TabNetClassifier

# Transform data using the existing preprocessor
X_train_transformed = preprocessor.fit_transform(X_train)

tabnet_model = TabNetClassifier(seed=RANDOM_STATE)

tabnet_model.fit(
    X_train_transformed,
    y_train.values,
    eval_set=[(X_train_transformed, y_train.values)],
    eval_metric=["auc"],
    max_epochs=50,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128
)

tabnet_probs = tabnet_model.predict_proba(X_train_transformed)[:, 1]

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.36252 | val_0_auc: 0.58924 |  0:00:06s
epoch 1  | loss: 0.34518 | val_0_auc: 0.61338 |  0:00:13s
epoch 2  | loss: 0.3426  | val_0_auc: 0.62202 |  0:00:19s
epoch 3  | loss: 0.34161 | val_0_auc: 0.62149 |  0:00:26s
epoch 4  | loss: 0.34177 | val_0_auc: 0.62584 |  0:00:32s
epoch 5  | loss: 0.34043 | val_0_auc: 0.62637 |  0:00:38s
epoch 6  | loss: 0.3408  | val_0_auc: 0.62437 |  0:00:57s
epoch 7  | loss: 0.34034 | val_0_auc: 0.6269  |  0:01:06s
epoch 8  | loss: 0.34045 | val_0_auc: 0.62952 |  0:01:13s
epoch 9  | loss: 0.3405  | val_0_auc: 0.62332 |  0:01:19s
epoch 10 | loss: 0.34161 | val_0_auc: 0.62865 |  0:01:26s
epoch 11 | loss: 0.34029 | val_0_auc: 0.63155 |  0:01:32s
epoch 12 | loss: 0.3405  | val_0_auc: 0.6307  |  0:01:39s
epoch 13 | loss: 0.3394  | val_0_auc: 0.6334  |  0:01:45s
epoch 14 | loss: 0.33901 | val_0_auc: 0.6327  |  0:01:52s
epoch 15 | loss: 0.33862 | val_0_auc: 0.6285  |  0:01:58s
epoch 16 | loss: 0.33938 | val_0_auc: 0.62991 |  0:02:05s
epoch 17 | los

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Transform data using the existing preprocessor
X_train_transformed = preprocessor.transform(X_train)

dnn_model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_transformed.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

dnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

dnn_model.fit(
    X_train_transformed,
    y_train,
    epochs=20,
    batch_size=256,
    verbose=1
)

dnn_probs = dnn_model.predict(X_train_transformed).ravel()

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


318/318 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - AUC: 0.6001 - loss: 0.3534
Epoch 2/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6186 - loss: 0.3423
Epoch 3/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.6268 - loss: 0.3406
Epoch 4/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - AUC: 0.6300 - loss: 0.3390
Epoch 5/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6338 - loss: 0.3380
Epoch 6/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - AUC: 0.6354 - loss: 0.3375
Epoch 7/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6363 - loss: 0.3374
Epoch 8/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6366 - loss: 0.3372
Epoch 9/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - AUC: 0.6402 - loss: 0.3366
Epoch 10/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.6402 - loss: 0.3363
Epoch 11/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.6429 - loss: 0.3357
Epoch 12/20
318/318 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - AUC: 0.6441 - loss: 0.3354
Epoch 13/20
318/318 ━━━━━━━━━━━━━━━━

In [21]:
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save classical models
filename_map = {
    "Logistic Regression"  : "pipeline_lr.pkl",
    "Decision Tree"        : "pipeline_dt.pkl",
    "Random Forest"        : "pipeline_rf.pkl",
    "HistGradientBoosting" : "pipeline_hgb.pkl",
    "XGBoost"              : "pipeline_xgb.pkl",
    "LightGBM"             : "pipeline_lgbm.pkl",
    "extra_trees"          : "pipeline_et.pkl",
    "gradient_boosting"    : "pipeline_gb.pkl",
    "svm"                  : "pipeline_svm.pkl",
    "catboost"             : "pipeline_cat.pkl"
}

for name, pipeline in models.items():
    path = os.path.join(MODEL_DIR, filename_map[name])
    joblib.dump(pipeline, path)

# Save TabNet
tabnet_model.save_model(os.path.join(MODEL_DIR, "tabnet_model"))

# Save Keras DNN
dnn_model.save(os.path.join(MODEL_DIR, "dnn_model.keras"))

# Save test set
X_test.to_csv(os.path.join(MODEL_DIR, "X_test.csv"), index=False)
y_test.to_csv(os.path.join(MODEL_DIR, "y_test.csv"), index=False)

print(f"All artifacts (Classical + DL) saved to: {MODEL_DIR}")

Successfully saved model at /content/models/tabnet_model.zip
All artifacts (Classical + DL) saved to: /content/models


In [23]:
from pytorch_tabnet.tab_model import TabNetClassifier

# Fix: Transform raw X_train into numeric features using the preprocessor
X_train_transformed = preprocessor.transform(X_train)

tabnet_model = TabNetClassifier(seed=RANDOM_STATE)

tabnet_model.fit(
    X_train_transformed,
    y_train.values,
    eval_set=[(X_train_transformed, y_train.values)],
    eval_metric=["auc"],
    max_epochs=50,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128
)

tabnet_probs = tabnet_model.predict_proba(X_train_transformed)[:, 1]

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.36252 | val_0_auc: 0.58924 |  0:00:20s
epoch 1  | loss: 0.34518 | val_0_auc: 0.61338 |  0:00:26s
epoch 2  | loss: 0.3426  | val_0_auc: 0.62202 |  0:00:39s
epoch 3  | loss: 0.34161 | val_0_auc: 0.62149 |  0:00:47s
epoch 4  | loss: 0.34177 | val_0_auc: 0.62584 |  0:00:53s
epoch 5  | loss: 0.34043 | val_0_auc: 0.62637 |  0:01:01s
epoch 6  | loss: 0.3408  | val_0_auc: 0.62437 |  0:01:07s
epoch 7  | loss: 0.34034 | val_0_auc: 0.6269  |  0:01:15s
epoch 8  | loss: 0.34045 | val_0_auc: 0.62952 |  0:01:22s
epoch 9  | loss: 0.3405  | val_0_auc: 0.62332 |  0:01:29s
epoch 10 | loss: 0.34161 | val_0_auc: 0.62865 |  0:01:37s
epoch 11 | loss: 0.34029 | val_0_auc: 0.63155 |  0:01:44s
epoch 12 | loss: 0.3405  | val_0_auc: 0.6307  |  0:01:51s
epoch 13 | loss: 0.3394  | val_0_auc: 0.6334  |  0:01:58s
epoch 14 | loss: 0.33901 | val_0_auc: 0.6327  |  0:02:05s
epoch 15 | loss: 0.33862 | val_0_auc: 0.6285  |  0:02:12s
epoch 16 | loss: 0.33938 | val_0_auc: 0.62991 |  0:02:19s
epoch 17 | los

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [25]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score

# Prepare containers for results
results = []

# 1. Evaluate Classical Models
for name, pipeline in models.items():
    y_prob = pipeline.predict_proba(X_train)[:, 1]
    y_pred = pipeline.predict(X_train)
    auc = roc_auc_score(y_train, y_prob)
    f1 = f1_score(y_train, y_pred)
    results.append({"Model": name, "Train ROC-AUC": auc, "Train F1 (pos)": f1})

# 2. Evaluate Keras DNN
# Pre-transform X_train for deep learning models
X_train_transformed = preprocessor.transform(X_train)

dnn_probs = dnn_model.predict(X_train_transformed, verbose=0).ravel()
dnn_preds = (dnn_probs > 0.5).astype(int)
results.append({
    "Model": "Keras DNN",
    "Train ROC-AUC": roc_auc_score(y_train, dnn_probs),
    "Train F1 (pos)": f1_score(y_train, dnn_preds)
})

# 3. Evaluate TabNet
tabnet_probs = tabnet_model.predict_proba(X_train_transformed)[:, 1]
tabnet_preds = tabnet_model.predict(X_train_transformed)
results.append({
    "Model": "TabNet",
    "Train ROC-AUC": roc_auc_score(y_train, tabnet_probs),
    "Train F1 (pos)": f1_score(y_train, tabnet_preds)
})

# Create and display DataFrame
results_df = pd.DataFrame(results).sort_values(by="Train ROC-AUC", ascending=False)
print("Final Model Comparison (In-Sample Performance):")
display(results_df.style.format({"Train ROC-AUC": "{:.4f}", "Train F1 (pos)": "{:.4f}"}))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Final Model Comparison (In-Sample Performance):


,Model,Train ROC-AUC,Train F1 (pos)
6,extra_trees,1.0000,0.9998
2,Random Forest,0.7935,0.3799
5,LightGBM,0.7410,0.3141
4,XGBoost,0.7108,0.2943
3,HistGradientBoosting,0.6856,0.2782
9,catboost,0.6772,0.0427
10,Keras DNN,0.6721,0.0109
7,gradient_boosting,0.6568,0.0263
1,Decision Tree,0.6440,0.2598
0,Logistic Regression,0.6425,0.2535


In [26]:
!zip -r /content/models_colab.zip /content/models_colab
print("Archive created at /content/models_colab.zip")

  adding: content/models_colab/ (stored 0%)
  adding: content/models_colab/catboost_info/ (stored 0%)
  adding: content/models_colab/catboost_info/catboost_training.json (deflated 76%)
  adding: content/models_colab/catboost_info/tmp/ (stored 0%)
  adding: content/models_colab/catboost_info/learn/ (stored 0%)
  adding: content/models_colab/catboost_info/learn/events.out.tfevents (deflated 78%)
  adding: content/models_colab/catboost_info/learn_error.tsv (deflated 57%)
  adding: content/models_colab/catboost_info/time_left.tsv (deflated 49%)
  adding: content/models_colab/models/ (stored 0%)
  adding: content/models_colab/models/pipeline_lgbm.pkl (deflated 60%)
  adding: content/models_colab/models/pipeline_hgb.pkl (deflated 61%)
  adding: content/models_colab/models/pipeline_svm.pkl (deflated 87%)
  adding: content/models_colab/models/pipeline_xgb.pkl (deflated 71%)
  adding: content/models_colab/models/X_test.csv (deflated 81%)
  adding: content/models_colab/models/dnn_model.keras (de